# Model Training s MLflow Tracking

**Trénovanie modelov s MLflow experimentom tracking**

V tomto notebooku:
1. MLflow setup a experiment tracking
2. Baseline model training
3. Hyperparameter tuning
4. Model comparison
5. Model registration do MLflow Model Registry

In [ ]:
# Imports
import os
import sys
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import xgboost as xgb

# MLflow
import mlflow
import mlflow.xgboost
from mlflow.tracking import MlflowClient

# Add src to path
sys.path.insert(0, os.path.abspath('../src'))

from data.preprocess import (
    build_preprocessing_pipeline,
    load_data
)

print("✅ Imports successful")
print(f"   MLflow: {mlflow.__version__}")
print(f"   XGBoost: {xgb.__version__}")

## 1. MLflow Setup

In [ ]:
# MLflow configuration
MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", "http://localhost:5000")
EXPERIMENT_NAME = "house-prices-local-experiments"

# Set tracking URI
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

# Create or get experiment
try:
    experiment_id = mlflow.create_experiment(EXPERIMENT_NAME)
    print(f"✅ Created experiment: {EXPERIMENT_NAME}")
except:
    experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
    experiment_id = experiment.experiment_id
    print(f"✅ Using existing experiment: {EXPERIMENT_NAME}")

mlflow.set_experiment(EXPERIMENT_NAME)

print(f"   Tracking URI: {MLFLOW_TRACKING_URI}")
print(f"   Experiment ID: {experiment_id}")

## 2. Načítanie a Preprocessing Dát

In [ ]:
# Load data
TRAIN_PATH = "../data/train.csv"

X, y = load_data(TRAIN_PATH)

# Train/validation/test split
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.176, random_state=42  # 0.176 * 0.85 ≈ 0.15
)

print(f"Train: {X_train.shape}")
print(f"Validation: {X_val.shape}")
print(f"Test: {X_test.shape}")

In [ ]:
# Preprocessing
pipeline = build_preprocessing_pipeline()

X_train_processed = pipeline.fit_transform(X_train)
X_val_processed = pipeline.transform(X_val)
X_test_processed = pipeline.transform(X_test)

print(f"Processed shapes:")
print(f"  Train: {X_train_processed.shape}")
print(f"  Validation: {X_val_processed.shape}")
print(f"  Test: {X_test_processed.shape}")

## 3. Helper Functions

In [ ]:
def compute_metrics(y_true, y_pred):
    """Compute regression metrics."""
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    rmsle = np.sqrt(mean_squared_error(y_true, y_pred))  # y is already log-transformed
    
    return {
        "rmse": rmse,
        "mae": mae,
        "r2": r2,
        "rmsle": rmsle
    }

def train_and_log_model(params, run_name=None):
    """Train model and log to MLflow."""
    with mlflow.start_run(run_name=run_name) as run:
        # Log parameters
        mlflow.log_params(params)
        
        # Train model
        model = xgb.XGBRegressor(
            **params,
            random_state=42,
            early_stopping_rounds=50,
            eval_metric="rmse"
        )
        
        model.fit(
            X_train_processed,
            y_train,
            eval_set=[(X_val_processed, y_val)],
            verbose=False
        )
        
        # Predictions
        y_pred_train = model.predict(X_train_processed)
        y_pred_val = model.predict(X_val_processed)
        y_pred_test = model.predict(X_test_processed)
        
        # Metrics
        train_metrics = compute_metrics(y_train, y_pred_train)
        val_metrics = compute_metrics(y_val, y_pred_val)
        test_metrics = compute_metrics(y_test, y_pred_test)
        
        # Log metrics
        mlflow.log_metrics({f"train_{k}": v for k, v in train_metrics.items()})
        mlflow.log_metrics({f"val_{k}": v for k, v in val_metrics.items()})
        mlflow.log_metrics({f"test_{k}": v for k, v in test_metrics.items()})
        
        # Log model
        signature = mlflow.models.infer_signature(X_val_processed, y_pred_val)
        mlflow.xgboost.log_model(
            model,
            artifact_path="model",
            signature=signature
        )
        
        # Tags
        mlflow.set_tags({
            "framework": "xgboost",
            "dataset": "kaggle-house-prices",
            "experiment_type": "local"
        })
        
        print(f"✅ Run ID: {run.info.run_id}")
        print(f"   Val RMSE: {val_metrics['rmse']:.4f}")
        print(f"   Test RMSE: {test_metrics['rmse']:.4f}")
        print(f"   Test R²: {test_metrics['r2']:.4f}")
        
        return model, run.info.run_id, test_metrics

print("✅ Helper functions defined")

## 4. Baseline Model

In [ ]:
# Baseline parameters
baseline_params = {
    "n_estimators": 100,
    "max_depth": 6,
    "learning_rate": 0.1,
    "subsample": 1.0,
    "colsample_bytree": 1.0,
    "min_child_weight": 1,
    "reg_alpha": 0,
    "reg_lambda": 1
}

print("Training baseline model...")
baseline_model, baseline_run_id, baseline_metrics = train_and_log_model(
    baseline_params,
    run_name="baseline-xgboost"
)

## 5. Hyperparameter Tuning Experiments

In [ ]:
# Experiment 1: More trees
exp1_params = baseline_params.copy()
exp1_params.update({
    "n_estimators": 500,
    "learning_rate": 0.05
})

print("\nExperiment 1: More trees + lower LR")
exp1_model, exp1_run_id, exp1_metrics = train_and_log_model(
    exp1_params,
    run_name="exp1-more-trees"
)

In [ ]:
# Experiment 2: Regularization
exp2_params = exp1_params.copy()
exp2_params.update({
    "reg_alpha": 0.1,
    "reg_lambda": 1.0,
    "subsample": 0.8,
    "colsample_bytree": 0.8
})

print("\nExperiment 2: Regularization")
exp2_model, exp2_run_id, exp2_metrics = train_and_log_model(
    exp2_params,
    run_name="exp2-regularization"
)

In [ ]:
# Experiment 3: Deeper trees
exp3_params = exp2_params.copy()
exp3_params.update({
    "max_depth": 8,
    "min_child_weight": 3
})

print("\nExperiment 3: Deeper trees")
exp3_model, exp3_run_id, exp3_metrics = train_and_log_model(
    exp3_params,
    run_name="exp3-deeper-trees"
)

## 6. Model Comparison

In [ ]:
# Compare all experiments
results = pd.DataFrame([
    {"model": "Baseline", **baseline_metrics},
    {"model": "Exp1 (More Trees)", **exp1_metrics},
    {"model": "Exp2 (Regularization)", **exp2_metrics},
    {"model": "Exp3 (Deeper Trees)", **exp3_metrics}
])

print("\n" + "="*60)
print("MODEL COMPARISON")
print("="*60)
print(results.to_string(index=False))
print("="*60)

In [ ]:
# Visualize comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# RMSE comparison
results.plot(x='model', y='rmse', kind='bar', ax=axes[0], legend=False)
axes[0].set_title('Test RMSE Comparison')
axes[0].set_ylabel('RMSE')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=45)

# R² comparison
results.plot(x='model', y='r2', kind='bar', ax=axes[1], legend=False, color='green')
axes[1].set_title('Test R² Comparison')
axes[1].set_ylabel('R²')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 7. Best Model Selection

In [ ]:
# Select best model based on test RMSE
best_idx = results['rmse'].idxmin()
best_model_name = results.loc[best_idx, 'model']
best_rmse = results.loc[best_idx, 'rmse']
best_r2 = results.loc[best_idx, 'r2']

print(f"\n🏆 Best Model: {best_model_name}")
print(f"   Test RMSE: {best_rmse:.4f}")
print(f"   Test R²: {best_r2:.4f}")

# Map to run ID
run_id_map = {
    "Baseline": baseline_run_id,
    "Exp1 (More Trees)": exp1_run_id,
    "Exp2 (Regularization)": exp2_run_id,
    "Exp3 (Deeper Trees)": exp3_run_id
}

best_run_id = run_id_map[best_model_name]
print(f"   Run ID: {best_run_id}")

## 8. Model Registration (Optional)

Registruj best model do MLflow Model Registry.

In [ ]:
# Register best model
MODEL_NAME = "house-price-xgboost"

try:
    model_uri = f"runs:/{best_run_id}/model"
    
    # Register model
    model_version = mlflow.register_model(
        model_uri=model_uri,
        name=MODEL_NAME
    )
    
    print(f"\n✅ Model registered:")
    print(f"   Name: {MODEL_NAME}")
    print(f"   Version: {model_version.version}")
    print(f"   Run ID: {best_run_id}")
    
    # Transition to Staging
    client = MlflowClient()
    client.transition_model_version_stage(
        name=MODEL_NAME,
        version=model_version.version,
        stage="Staging"
    )
    print(f"   Stage: Staging")
    
except Exception as e:
    print(f"❌ Model registration failed: {e}")
    print("   This is OK if MLflow server is not running.")
    print("   Start MLflow server: mlflow ui")

## 9. Summary

### Čo sme dosiahli:

1. ✅ **MLflow Tracking Setup**
   - Experiment tracking
   - Automatic metric logging
   - Model artifacts storage

2. ✅ **Model Training**
   - Baseline model
   - 3 hyperparameter experiments
   - Systematic comparison

3. ✅ **Model Selection**
   - Best model identifikovaný
   - Registered to MLflow Model Registry
   - Ready for deployment

### Next Steps:

1. **SageMaker Training** (Krok 4):
   - Deploy training na SageMaker
   - Remote MLflow tracking
   - Distributed training

2. **Model Deployment** (Krok 5):
   - Deploy na SageMaker endpoint
   - Staging → Production workflow

3. **CI/CD** (Krok 6):
   - Automated training pipeline
   - Automated deployment

## 10. MLflow UI

Skontroluj MLflow UI pre všetky experimenty:

```bash
# Start MLflow UI (if not running)
mlflow ui --port 5000
```

Otvor v prehliadači: http://localhost:5000